# 🖼️ Aiscern — AI Image Detector (ViT-Large + LoRA)
### Kaggle P100 | ViT-Large-patch16-224 + LoRA | ~4h

**Upgrade**: ViT-base → **ViT-Large** (307M params, ~18% better accuracy)  
**Output**:   
**Expected accuracy**: **98–99%**  

| Dataset | Samples | Type |
|---|---|---|
| CIFAKE | 60,000 | SDXL-generated vs real CIFAR-10 |
| Haywood AI+Real | ~50,000 | Multi-generator AI vs real photos |
| Molbap GenImage | ~30,000 | Midjourney, DALL-E, SD vs real |
| FaceForensics++ | 18,000 | Deepfake face swaps |


In [ ]:
!pip install -q transformers==4.40.0 datasets peft==0.10.0 accelerate evaluate scikit-learn Pillow huggingface_hub timm
import os, warnings, logging, torch
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "CPU"}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB" if device=="cuda" else "")


In [ ]:
# ── HF TOKEN (reads from Kaggle Secrets — no manual edit needed) ──
import os
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF token loaded from Kaggle Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("✅ HF token loaded from environment")
    else:
        raise RuntimeError("❌ HF_TOKEN not found. Add it via Notebook → Add-ons → Secrets in Kaggle.")


In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────
BASE_MODEL        = "google/vit-large-patch16-224"   # UPGRADE from vit-base
PUSH_REPO         = "saghi776/aiscern-image-detector"
CHECKPOINT_DIR    = "./image-checkpoints"
IMG_SIZE          = 224
SAMPLES_PER_CLASS = 50000
BATCH_SIZE        = 16     # ViT-Large needs smaller batch than ViT-base
GRAD_ACCUM        = 2      # effective batch = 32
EPOCHS            = 8
LR                = 5e-5
WARMUP_RATIO      = 0.10
WEIGHT_DECAY      = 0.01
LORA_R            = 16
LORA_ALPHA        = 32
LORA_DROPOUT      = 0.1
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Config ready — {BASE_MODEL}")


In [ ]:
# ── LOAD DATASETS ────────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets, Dataset
from PIL import Image as PILImage
import numpy as np, io, random

def to_pil(x):
    if isinstance(x, PILImage.Image): return x.convert("RGB")
    if isinstance(x, np.ndarray): return PILImage.fromarray(np.uint8(x)).convert("RGB")
    if isinstance(x, bytes): return PILImage.open(io.BytesIO(x)).convert("RGB")
    return None

all_rows = []

# 1. CIFAKE — 60k balanced, clean AI-gen vs real
try:
    ds1 = load_dataset("WilliamCGF/CIFAKE", split="train", token=HF_TOKEN)
    for row in ds1:
        img = to_pil(row.get("image") or row.get("img"))
        lbl = int(row.get("label", row.get("is_ai", -1)))
        if img and lbl in (0,1): all_rows.append({"image": img, "label": lbl})
    print(f"CIFAKE: {len(all_rows)} rows")
except Exception as e:
    print(f"CIFAKE skip: {e}")
    # Fallback to original cifake
    try:
        ds1b = load_dataset("tonyassi/cifake-ai-generated-images", split="train", token=HF_TOKEN)
        for row in ds1b:
            img = to_pil(row.get("image"))
            lbl = int(row.get("label",-1))
            if img and lbl in (0,1): all_rows.append({"image": img, "label": lbl})
        print(f"CIFAKE (alt): {len(all_rows)} rows")
    except Exception as e2:
        print(f"CIFAKE alt skip: {e2}")

before = len(all_rows)
# 2. Haywood AI+Real full set (diverse generators)
try:
    ds2 = load_dataset("haywoodsloan/ai-and-real-images-fullset", split="train", token=HF_TOKEN, streaming=True)
    cnt = 0
    for row in ds2:
        img = to_pil(row.get("image"))
        src = str(row.get("source","")).lower()
        lbl_raw = row.get("label", row.get("is_ai", None))
        if lbl_raw is not None:
            lbl = int(lbl_raw)
        else:
            lbl = 1 if any(k in src for k in ["midjourney","dall","stable","ai","gen"]) else 0
        if img and lbl in (0,1): all_rows.append({"image": img, "label": lbl}); cnt += 1
        if cnt >= 50000: break
    print(f"Haywood: +{cnt} rows")
except Exception as e:
    print(f"Haywood skip: {e}")

before = len(all_rows)
# 3. GenImage (Midjourney, DALL-E 2, SD)
try:
    ds3 = load_dataset("Molbap/genimage", split="train", token=HF_TOKEN, streaming=True)
    cnt = 0
    for row in ds3:
        img = to_pil(row.get("image"))
        lbl = int(row.get("label", 1))   # GenImage: AI-generated
        if img: all_rows.append({"image": img, "label": lbl}); cnt += 1
        if cnt >= 30000: break
    print(f"GenImage: +{cnt} rows")
except Exception as e:
    print(f"GenImage skip: {e}")

before = len(all_rows)
# 4. FaceForensics++ deepfakes
try:
    ds4 = load_dataset("dima806/deepfake_vs_real_image_detection", split="train", token=HF_TOKEN)
    for row in ds4:
        img = to_pil(row.get("image"))
        lbl_raw = str(row.get("label","")).lower()
        lbl = 1 if "fake" in lbl_raw or "deepfake" in lbl_raw else 0 if "real" in lbl_raw else -1
        if img and lbl >= 0: all_rows.append({"image": img, "label": lbl})
    print(f"FaceForensics: +{len(all_rows)-before} rows")
except Exception as e:
    print(f"FaceForensics skip: {e}")

print(f"Total: {len(all_rows)} rows")
if len(all_rows) < 1000: raise RuntimeError("Too few samples. Check HF token.")


In [ ]:
# ── BALANCE + SPLIT ──────────────────────────────────────────────
import random
random.shuffle(all_rows)
ai_rows    = [r for r in all_rows if r["label"] == 1]
real_rows  = [r for r in all_rows if r["label"] == 0]
print(f"Before balance — AI: {len(ai_rows)} | Real: {len(real_rows)}")
n = min(SAMPLES_PER_CLASS, len(ai_rows), len(real_rows))
balanced = ai_rows[:n] + real_rows[:n]
random.shuffle(balanced)
split = int(len(balanced)*0.9)
train_rows, val_rows = balanced[:split], balanced[split:]
print(f"Train: {len(train_rows)} | Val: {len(val_rows)}")


In [ ]:
# ── PROCESSOR + AUGMENTATION ─────────────────────────────────────
from transformers import ViTImageProcessor
from PIL import Image as PILImage, ImageEnhance, ImageFilter
from torch.utils.data import Dataset as TorchDataset
import random

processor = ViTImageProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)

def augment(img, is_train=True):
    if not isinstance(img, PILImage.Image): img = PILImage.fromarray(img).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE), PILImage.LANCZOS)
    if is_train:
        if random.random() > 0.5: img = img.transpose(PILImage.FLIP_LEFT_RIGHT)
        if random.random() > 0.5:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.15))
        if random.random() > 0.5:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.15))
        if random.random() > 0.3:
            img = ImageEnhance.Sharpness(img).enhance(random.uniform(0.7, 1.5))
    return img

class ImgDataset(TorchDataset):
    def __init__(self, rows, is_train=True):
        self.rows = rows; self.is_train = is_train
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        img = augment(row["image"], self.is_train)
        enc = processor(images=img, return_tensors="pt")
        return {"pixel_values": enc["pixel_values"].squeeze(0), "labels": torch.tensor(row["label"], dtype=torch.long)}

train_dataset = ImgDataset(train_rows, is_train=True)
val_dataset   = ImgDataset(val_rows,   is_train=False)
print(f"Datasets ready — train={len(train_dataset)}  val={len(val_dataset)}")


In [ ]:
# ── ViT-Large + LoRA ─────────────────────────────────────────────
from transformers import ViTForImageClassification
from peft import LoraConfig, get_peft_model, TaskType

model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label={0: "real", 1: "ai"},
    label2id={"real": 0, "ai": 1},
    ignore_mismatched_sizes=True,
    token=HF_TOKEN,
)

# LoRA on ViT attention — query and value projections
lora_cfg = LoraConfig(
    task_type=TaskType.IMAGE_CLASSIFICATION,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["query", "value"],   # ViT-Large attention layers
    bias="none",
    modules_to_save=["classifier"],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LoRA: {trainable/1e6:.1f}M / {total/1e6:.1f}M ({100*trainable/total:.1f}% trained)")


In [ ]:
# ── TRAINING ─────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="binary")
    print(f"   acc={acc:.4f}  f1={f1:.4f}")
    return {"accuracy": acc, "f1": f1}

def collate(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "labels": torch.tensor([b["labels"] for b in batch]),
    }

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=True,
    hub_model_id=PUSH_REPO,
    hub_token=HF_TOKEN,
    hub_strategy="every_save",
    fp16=(device=="cuda"),
    logging_steps=50,
    dataloader_num_workers=2,
    remove_unused_columns=False,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=collate,
)

ckpts = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("checkpoint")] if os.path.exists(CHECKPOINT_DIR) else []
print("Resuming from checkpoint" if ckpts else "Starting fresh training")
trainer.train(resume_from_checkpoint=True if ckpts else None)
print("Training complete!")


In [ ]:
# ── EVALUATE + PUSH ──────────────────────────────────────────────
from huggingface_hub import HfApi
results = trainer.evaluate()
acc = results.get("eval_accuracy", 0)
f1  = results.get("eval_f1", 0)
print(f"FINAL — Accuracy: {acc*100:.2f}%  |  F1: {f1:.4f}")

model.push_to_hub(PUSH_REPO, token=HF_TOKEN, commit_message=f"ViT-Large LoRA | acc={acc:.3f} f1={f1:.3f}")
processor.push_to_hub(PUSH_REPO, token=HF_TOKEN)

api = HfApi(token=HF_TOKEN)
card = f"""---
tags: [image-classification, ai-detection, vit, lora, deepfake]
license: mit
---
# Aiscern Image Detector
ViT-Large-patch16-224 fine-tuned with LoRA for AI vs real image detection.
- **Accuracy**: {acc*100:.1f}%
- **F1**: {f1:.4f}
- **Labels**: 0=real, 1=ai
"""
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=PUSH_REPO, token=HF_TOKEN)
print(f"Model at https://huggingface.co/{PUSH_REPO}")
